# Week 4 — Baseline Action Score

**Assignment:** Week 4 — Signals, Baseline & Ranked Actions  
**Lane:** Refresh / content opportunity  
**Goal:** Audit two observable signals, encode one explainable baseline rule, generate a ranked action queue, and review the top 10 skeptically.

### Rule design

The baseline prioritizes pages that are:

- stale: `days_since_last_update >= 91`
- visibly ranking: `impressions_90d >= 500` and `0 < avg_position <= 20`
- low CTR: `ctr < 0.50`

The queue uses one reason code and one action label.

### Leakage guard

The target is `is_declining_label = (trend_direction == "down")`. `trend_direction`, `trend_pct`, and `is_declining_label` are **not used as rule inputs**. They are used only for signal auditing/evaluation.

The notebook reads only the local raw dataset and regenerates the required CSV on every run.

## 0. Setup and data loading

The raw dataset is stored at:

`data/raw/content_refresh_anonymized.csv`

The output CSV is intentionally written under `work/outputs/` and should remain uncommitted because the repository's CI leak guard blocks data files.

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()

# Make the notebook robust when launched from either the repo root
# or the work/notebooks directory.
if (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
    OUTPUT_PATH = ROOT / "work/outputs/baseline_action_score.csv"
elif (ROOT.parent.parent / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = ROOT.parent.parent
    DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
    OUTPUT_PATH = ROOT / "work/outputs/baseline_action_score.csv"
else:
    raise FileNotFoundError(
        "Could not locate data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from the repository."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Data path: {DATA_PATH}")
print(f"Output path: {OUTPUT_PATH}")

required_columns = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction",
]

missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("\nRequired columns verified.")

Dataset shape: (30000, 44)
Data path: c:\Users\parve\OneDrive\Desktop\Zahed\assignment\zahed-flyrank-ml\data\raw\content_refresh_anonymized.csv
Output path: c:\Users\parve\OneDrive\Desktop\Zahed\assignment\zahed-flyrank-ml\work\outputs\baseline_action_score.csv

Required columns verified.


## 1. Signal checks

### Signal 1 — Staleness

**Signal:** `days_since_last_update`

This signal is linked to the refresh/staleness logic discussed in the session. The audit checks whether observed staleness buckets show different declining rates.

The verdict is based on the observed bucket pattern, not on the desired outcome.

In [21]:
# Use the existing freshness buckets where available.
staleness_audit = (
    df.groupby("freshness_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_position=("avg_position", "median"),
          median_ctr=("ctr", "median"),
          declining_rate=(
              "trend_direction",
              lambda x: x.eq("down").mean()
          ),
      )
      .reset_index()
)

# Sort naturally by the numeric start of the freshness bucket.
def freshness_sort_key(value):
    text = str(value)
    try:
        return int(text.split("-")[0].replace("+", ""))
    except ValueError:
        return 999999

staleness_audit["_sort"] = staleness_audit["freshness_tier"].map(freshness_sort_key)
staleness_audit = (
    staleness_audit
    .sort_values("_sort")
    .drop(columns="_sort")
    .reset_index(drop=True)
)

display(staleness_audit)

print(f"\nTotal audited rows: {staleness_audit['n'].sum():,}")

,freshness_tier,n,median_impressions,median_position,median_ctr,declining_rate
0,0-30,20480,470.0,9.9,0.04,0.511377
1,31-90,175,510.0,13.9,0.00,0.588571
2,91-180,9171,1692.0,13.6,0.10,0.611057
3,181+,174,15.5,7.0,0.00,0.471264



Total audited rows: 30,000


### Staleness verdict — CONFIRMED

The 91–180 day bucket has a higher declining rate than the 0–30 day bucket:

- 0–30 days: **51.1%**
- 91–180 days: **61.1%**

This is a directional association, not a causal claim. The 181+ bucket is small and does not continue a strictly monotonic pattern, so the signal should not be described as universally monotonic.

**Verdict: CONFIRMED.**

### Signal 2 — CTR relative to position

The second signal is not CTR alone. It is the combination of:

- `ctr`
- `avg_position`
- `impressions_90d`

The motivation is the CTR-vs-position logic behind the CTR-fix flag: a low CTR is more actionable when a page already has meaningful search visibility and ranks within the first 20 positions.

In [22]:
df["ctr_position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-0.001, 3, 10, 20, 50, float("inf")],
    labels=["top_3", "4-10", "11-20", "21-50", "50+"],
)

ctr_position_audit = (
    df.groupby("ctr_position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_impressions=("impressions_90d", "median"),
          declining_rate=(
              "trend_direction",
              lambda x: x.eq("down").mean()
          ),
      )
      .reset_index()
)

display(ctr_position_audit)

df["ctr_visibility_bucket"] = pd.cut(
    df["ctr"],
    bins=[-0.001, 0.05, 0.10, 0.25, 0.50, float("inf")],
    labels=["0-0.05", "0.05-0.10", "0.10-0.25", "0.25-0.50", "0.50+"],
)

ctr_audit = (
    df.groupby("ctr_visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_position=("avg_position", "median"),
          median_impressions=("impressions_90d", "median"),
          declining_rate=(
              "trend_direction",
              lambda x: x.eq("down").mean()
          ),
      )
      .reset_index()
)

display(ctr_audit)

,ctr_position_bucket,n,median_ctr,median_impressions,declining_rate
0,top_3,2346,0.00,3.0,0.245524
1,4-10,11842,0.16,1184.0,0.569414
2,11-20,7273,0.10,870.0,0.609515
3,21-50,7225,0.03,807.0,0.561799
4,50+,1314,0.00,219.5,0.343227


,ctr_visibility_bucket,n,median_position,median_impressions,declining_rate
0,0-0.05,14419,12.5,104.0,0.515223
1,0.05-0.10,2077,13.8,3279.0,0.652865
2,0.10-0.25,5266,10.9,2949.0,0.602165
3,0.25-0.50,4089,9.1,2880.0,0.561506
4,0.50+,4149,7.9,985.0,0.484454


### CTR/position verdict — CONFIRMED

The position audit shows elevated declining rates in the 4–20 range:

- 4–10: **56.9%**
- 11–20: **61.0%**

The CTR audit also shows elevated declining rates in the low-to-moderate CTR range, especially 0.05–0.10 (**65.3%**).

Because the relationship is contextual rather than monotonic, the baseline uses CTR together with visibility and position rather than using CTR alone.

**Verdict: CONFIRMED.**

## 2. Baseline rule

### Candidate comparison

Three rule candidates were checked during development:

| Candidate | n | Coverage | Declining rate |
|---|---:|---:|---:|
| A — stale + visible + CTR < 0.10 | 1,139 | 3.80% | 75.42% |
| **B — stale + visible + CTR < 0.50** | **3,534** | **11.78%** | **65.62%** |
| C — stale + visible | 4,350 | 14.50% | 62.11% |

Candidate B was selected because it provides a broader queue while retaining a substantial concentration of declining pages.

### Final eligibility rule

A page enters the baseline queue when:

```text
days_since_last_update >= 91
AND impressions_90d >= 500
AND 0 < avg_position <= 20
AND ctr < 0.50
```

### One reason code

`stale_visible_low_ctr`

### One action label

`refresh_and_review_ctr`

The target is not used in the score or eligibility rule.

In [23]:
# Final Week-4 eligibility rule.
eligible = (
    (df["days_since_last_update"] >= 91)
    & (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.50)
)

print(f"Eligible pages: {eligible.sum():,}")
print(f"Coverage: {eligible.mean():.2%}")

observed_declining_rate = df.loc[eligible, "trend_direction"].eq("down").mean()
overall_declining_rate = df["trend_direction"].eq("down").mean()

print(f"Observed declining rate among eligible pages: {observed_declining_rate:.2%}")
print(f"Overall declining rate: {overall_declining_rate:.2%}")
print(
    f"Observed lift over overall rate: "
    f"{observed_declining_rate - overall_declining_rate:+.2%}"
)

Eligible pages: 3,534
Coverage: 11.78%
Observed declining rate among eligible pages: 65.62%
Overall declining rate: 54.21%
Observed lift over overall rate: +11.41%


### Score

For eligible pages, the score ranks the same single rule's candidates by three observable opportunity components:

- **40% staleness:** more days since update increases priority; 91 days maps to 0 and 180+ maps to 1.
- **35% CTR opportunity:** lower CTR gives higher priority within the `< 0.50` eligibility range.
- **25% position opportunity:** positions closer to 1 receive higher priority within the eligible 1–20 range.

This is a deterministic ranking baseline, not a learned model.

In [24]:
queue = df.loc[eligible].copy()

# 1. Staleness: 91 days is the eligibility threshold; 180+ is capped.
queue["staleness_score"] = (
    (queue["days_since_last_update"] - 91) / (180 - 91)
).clip(0, 1)

# 2. CTR opportunity: lower CTR is higher opportunity.
queue["ctr_opportunity_score"] = (
    1 - queue["ctr"] / 0.50
).clip(0, 1)

# 3. Position opportunity: better eligible positions get higher priority.
queue["position_opportunity_score"] = (
    1 - (queue["avg_position"] - 1) / 19
).clip(0, 1)

# One deterministic baseline score.
queue["baseline_action_score"] = (
    0.40 * queue["staleness_score"]
    + 0.35 * queue["ctr_opportunity_score"]
    + 0.25 * queue["position_opportunity_score"]
).clip(0, 1)

# Exactly one reason code and one action label.
queue["reason_code"] = "stale_visible_low_ctr"
queue["action_label"] = "refresh_and_review_ctr"

# Deterministic tie-breaking.
queue = (
    queue
    .sort_values(
        ["baseline_action_score", "content_id"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

queue["baseline_rank"] = np.arange(1, len(queue) + 1)

print(f"Ranked queue size: {len(queue):,}")
display(
    queue[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "reason_code",
            "action_label",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
        ]
    ].head(10)
)

Ranked queue size: 3,534


,baseline_rank,content_id,baseline_action_score,reason_code,action_label,days_since_last_update,impressions_90d,avg_position,ctr
0,1,content_72496874f806,0.768842,stale_visible_low_ctr,refresh_and_review_ctr,301,821,5.8,0.24
1,2,content_928af3e22c80,0.721263,stale_visible_low_ctr,refresh_and_review_ctr,193,1697,15.8,0.12
2,3,content_e3ff1b093148,0.714526,stale_visible_low_ctr,refresh_and_review_ctr,183,1408,7.8,0.28
3,4,content_7e97f31ba9bb,0.658427,stale_visible_low_ctr,refresh_and_review_ctr,104,603,0.7,0.00
4,5,content_d0351144852b,0.658427,stale_visible_low_ctr,refresh_and_review_ctr,104,884,0.6,0.00
5,6,content_4bb993e9270e,0.654480,stale_visible_low_ctr,refresh_and_review_ctr,104,1986,1.3,0.00
6,7,content_6226ee6adc91,0.652947,stale_visible_low_ctr,refresh_and_review_ctr,183,545,17.8,0.18
7,8,content_97a08f7b106c,0.651848,stale_visible_low_ctr,refresh_and_review_ctr,104,1108,1.5,0.00
8,9,content_fb4884e34467,0.651848,stale_visible_low_ctr,refresh_and_review_ctr,104,580,1.5,0.00
9,10,content_a37ce8ddc090,0.650310,stale_visible_low_ctr,refresh_and_review_ctr,106,1039,2.3,0.00


In [25]:
# Write the required queue.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
]

queue[output_columns].to_csv(OUTPUT_PATH, index=False)

print(f"Wrote: {OUTPUT_PATH}")
print(f"Rows written: {len(queue):,}")
print(f"Columns written: {len(output_columns)}")

Wrote: c:\Users\parve\OneDrive\Desktop\Zahed\assignment\zahed-flyrank-ml\work\outputs\baseline_action_score.csv
Rows written: 3,534
Columns written: 9


## 3. Top-10 review

The review is intentionally skeptical. Each row states why the rule selected it and what could make the recommendation wrong.

Client identifiers are excluded from the review because this work may be public.

In [26]:
top10 = queue.head(10).copy()

review_rows = []

for rank, row in top10.iterrows():
    reasons = (
        f"{int(row['days_since_last_update'])} days stale, "
        f"{row['impressions_90d']:.0f} 90-day impressions, "
        f"position {row['avg_position']:.1f}, "
        f"CTR {row['ctr']:.2f}%"
    )

    if row["ctr"] == 0:
        wrong = (
            "Zero clicks can reflect query intent, SERP features, "
            "or measurement behavior rather than a content-refresh problem."
        )
    elif pd.isna(row.get("search_volume", np.nan)):
        wrong = (
            "Missing search-volume context and sparse clicks could make "
            "the low-CTR signal noisy."
        )
    else:
        wrong = (
            "The observed low CTR may have an explanation other than "
            "stale content, so the recommendation requires human review."
        )

    review_rows.append(
        {
            "rank": int(row["baseline_rank"]),
            "action": row["action_label"],
            "why_its_here": reasons,
            "what_would_make_it_wrong": wrong,
        }
    )

top10_review = pd.DataFrame(review_rows)
display(top10_review)

,rank,action,why_its_here,what_would_make_it_wrong
0,1,refresh_and_review_ctr,"301 days stale, 821 90-day impressions, positi...",Missing search-volume context and sparse click...
1,2,refresh_and_review_ctr,"193 days stale, 1697 90-day impressions, posit...",The observed low CTR may have an explanation o...
2,3,refresh_and_review_ctr,"183 days stale, 1408 90-day impressions, posit...",The observed low CTR may have an explanation o...
3,4,refresh_and_review_ctr,"104 days stale, 603 90-day impressions, positi...","Zero clicks can reflect query intent, SERP fea..."
4,5,refresh_and_review_ctr,"104 days stale, 884 90-day impressions, positi...","Zero clicks can reflect query intent, SERP fea..."
5,6,refresh_and_review_ctr,"104 days stale, 1986 90-day impressions, posit...","Zero clicks can reflect query intent, SERP fea..."
6,7,refresh_and_review_ctr,"183 days stale, 545 90-day impressions, positi...",The observed low CTR may have an explanation o...
7,8,refresh_and_review_ctr,"104 days stale, 1108 90-day impressions, posit...","Zero clicks can reflect query intent, SERP fea..."
8,9,refresh_and_review_ctr,"104 days stale, 580 90-day impressions, positi...","Zero clicks can reflect query intent, SERP fea..."
9,10,refresh_and_review_ctr,"106 days stale, 1039 90-day impressions, posit...","Zero clicks can reflect query intent, SERP fea..."


## 4. Weak picks / limitations

The top-ranked queue is intentionally treated as decision support rather than proof that a page needs a refresh.

### Observed weakness

Several top-ranked pages have very low or zero clicks while ranking highly. That creates a plausible CTR opportunity, but it can also be caused by:

- query intent
- SERP features
- measurement/tracking behavior
- sparse observations
- missing contextual fields

The rule also does not model every possible business consideration. It should therefore be reviewed by a human before action.

### Weak-pick check

A useful diagnostic is to inspect the lowest-scoring eligible pages. They satisfy the same rule but have less combined opportunity under the score.

In [27]:
weak_picks = (
    queue[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "reason_code",
            "action_label",
        ]
    ]
    .tail(10)
    .sort_values("baseline_rank")
)

display(weak_picks)

,baseline_rank,content_id,baseline_action_score,days_since_last_update,impressions_90d,avg_position,ctr,reason_code,action_label
3524,3525,content_a57b45c0ef3b,0.096111,104,3765,18.2,0.48,stale_visible_low_ctr,refresh_and_review_ctr
3525,3526,content_688d1be4cb88,0.095690,104,1424,17.7,0.49,stale_visible_low_ctr,refresh_and_review_ctr
3526,3527,content_ee3833c68977,0.090427,104,1030,18.1,0.49,stale_visible_low_ctr,refresh_and_review_ctr
3527,3528,content_3e3c3aacb713,0.088637,104,1277,19.3,0.47,stale_visible_low_ctr,refresh_and_review_ctr
3528,3529,content_9f17b1ec8d43,0.085585,104,11372,19.0,0.48,stale_visible_low_ctr,refresh_and_review_ctr
3529,3530,content_d777099643ca,0.083848,104,6374,18.6,0.49,stale_visible_low_ctr,refresh_and_review_ctr
3530,3531,content_4b05c87e3f08,0.079427,104,29516,20.0,0.47,stale_visible_low_ctr,refresh_and_review_ctr
3531,3532,content_08c28f769a4f,0.075059,104,1657,19.8,0.48,stale_visible_low_ctr,refresh_and_review_ctr
3532,3533,content_105115d87a31,0.065427,104,7342,20.0,0.49,stale_visible_low_ctr,refresh_and_review_ctr
3533,3534,content_1fd57c10efa8,0.065427,104,1432,20.0,0.49,stale_visible_low_ctr,refresh_and_review_ctr


## 5. Self-check

The following checks verify the Week-4 deliverable requirements.

In [28]:
# Self-checks

# 1. Output exists and has the expected number of rows.
assert OUTPUT_PATH.exists(), "Required CSV was not written."
assert len(queue) == int(eligible.sum()), "Queue size mismatch."

# 2. Exactly one reason code and one action label.
assert queue["reason_code"].nunique() == 1
assert queue["action_label"].nunique() == 1

# 3. Every queued row satisfies the single eligibility rule.
assert (
    (queue["days_since_last_update"] >= 91)
    & (queue["impressions_90d"] >= 500)
    & (queue["avg_position"] > 0)
    & (queue["avg_position"] <= 20)
    & (queue["ctr"] < 0.50)
).all()

# 4. No target-derived fields are used in the score.
score_inputs = {
    "days_since_last_update",
    "ctr",
    "avg_position",
}
forbidden_rule_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

assert score_inputs.isdisjoint(forbidden_rule_inputs)

# 5. Scores are bounded and ranks are unique.
assert queue["baseline_action_score"].between(0, 1).all()
assert queue["baseline_rank"].is_unique

# 6. Output schema exists.
written = pd.read_csv(OUTPUT_PATH)
assert list(written.columns) == [
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
]

print("PASS — all Week-4 self-checks completed.")
print(f"Queue rows: {len(written):,}")
print(f"Output: {OUTPUT_PATH}")

PASS — all Week-4 self-checks completed.
Queue rows: 3,534
Output: c:\Users\parve\OneDrive\Desktop\Zahed\assignment\zahed-flyrank-ml\work\outputs\baseline_action_score.csv


## Final summary

**Signal 1:** Staleness — **CONFIRMED**  
**Signal 2:** CTR/position opportunity — **CONFIRMED**

**Baseline rule:** stale + visible + low CTR

**Reason code:** `stale_visible_low_ctr`

**Action label:** `refresh_and_review_ctr`

**Target leakage:** `trend_direction`, `trend_pct`, and `is_declining_label` are not used as rule inputs.

**Output:** `work/outputs/baseline_action_score.csv`

The baseline is intentionally simple and explainable. Week 5's learned model should be evaluated against this baseline rather than replacing it without comparison.